# 08_finish_all - finish the final-token repair END TO END

One session, T4 GPU, **Runtime -> Run all**. Does everything still missing and
archives + downloads every artifact at the end so a dropped session never loses work.

| step | what | ~time |
|---|---|---|
| JOB A | generate the **full-A/D** (n=150) final-token causal rows for 4 branches (`--all-ad-sensitivity`) | 1.5-3 h |
| probe | confirm the StrongREJECT judge loads in fp16 (stops with the real error otherwise) | 3 min |
| JOB B | judge cross-fit + full-A/D with **StrongREJECT only** (`--skip-wildguard`); held-out carried from the overnight file | 20-40 min |
| POST | `confirmatory_behavioral_endpoints --condition-infix ft_` - fills `primary` + `cross_fitted` + `full_A_sensitivity` | 20 min |
| SAVE | dated Drive snapshot + tar.gz + sha256 manifest + browser download of a zip + git push | 2 min |

Needs a Colab `HF_TOKEN` secret (notebook access ON) on an account that accepted the
`google/gemma-2b` **and** `allenai/wildguard` licences.

**Checkpoints after every branch** (explicit Drive copy + git commit/push), so a
crash after JOB A resumes JOB A instantly and never re-generates a finished branch.


## 1. Clone + pin + Drive + bind + HF


In [ ]:
PINNED_COMMIT = "c174f5e3b4a4d8eac0183780ec2a8dcb397d942e"
import os, sys, subprocess, glob, json, shutil, datetime
from pathlib import Path
from collections import Counter

REPO = "https://github.com/urosavurdic/dpo-safety-representations.git"
if not os.path.isdir("dpo-safety-representations"):
    subprocess.run(["git", "clone", REPO], check=False)
os.chdir("dpo-safety-representations")
subprocess.run(["git", "fetch", "--all", "--quiet"], check=True)
BR = "agent/c-quadrant-end-to-end-e0e2317a"
subprocess.run(["git", "checkout", "-B", BR, "origin/" + BR], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BR], check=False)
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("HEAD  :", HEAD)
print("PINNED:", PINNED_COMMIT)
if PINNED_COMMIT not in ("REPLACE_AFTER_COMMIT", HEAD):
    print("  NOTE: branch tip moved past the pinned commit - fine if it is just later"
          " work; re-pull this notebook if a result looks wrong.")

from google.colab import drive, userdata
drive.mount("/content/drive")
cands = ["/content/drive/MyDrive/dpo_v2"] \
      + sorted(glob.glob("/content/drive/.shortcut-targets-by-id/*/dpo_v2")) \
      + sorted(glob.glob("/content/drive/Shareddrives/*/dpo_v2"))
REAL = next((c for c in cands if os.path.isdir(os.path.join(c, "results"))), None)
assert REAL, "no dpo_v2 folder found:\n  " + "\n  ".join(cands)
os.environ["DPO_DRIVE_ROOT"] = REAL
from src.colab_persist import bind
print(bind(persist_hf_cache=False))   # results/ -> shared Drive folder (auto-persist)

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF ok")

# ---- checkpoint helper: explicit Drive copy (symlink-independent) + best-effort git ----
ARCH = Path(REAL) / "final_token_repair_archives"; ARCH.mkdir(exist_ok=True)
FT_GLOBS = ["results/raw/causal_ablation_v2_*L24-28*finaltoken*.json",
            "results/refusal_direction/*_final_token*.npy",
            "results/refusal_direction/*_final_token*binding.json",
            "results/final_token_repair/directions/*.npy",
            "results/final_token_repair/bindings/*.json",
            "results/final_token_repair/summaries/*.json",
            "results/final_token_repair/manifests/*.json"]

def ckpt(label):
    fs = sorted({q for pat in FT_GLOBS for q in glob.glob(pat)
                 if os.path.isfile(q) and "judges/" not in q and "behavioral_judges_v2_" not in q})
    n = 0
    for f in fs:
        try:
            shutil.copy2(f, ARCH / Path(f).name); n += 1
        except Exception as ex:
            print("   drive-copy skip", f, ex)
    print("  [ckpt " + label + "] " + str(n) + " files -> " + str(ARCH))
    try:
        subprocess.run(["git", "config", "user.email", "noreply@anthropic.com"], check=True)
        subprocess.run(["git", "config", "user.name", "final-token-repair (Colab)"], check=True)
        subprocess.run(["git", "add", "--"] + fs, check=True, capture_output=True)
        if subprocess.check_output(["git", "diff", "--cached", "--name-only"], text=True).strip():
            subprocess.run(["git", "commit", "-m",
                            "final-token repair (Colab): " + label +
                            "\n\nCo-Authored-By: Claude Sonnet 5 <noreply@anthropic.com>"],
                           check=True, capture_output=True)
            subprocess.run(["git", "pull", "--rebase", "origin", BR], check=False, capture_output=True)
            r = subprocess.run(["git", "push", "origin", BR], capture_output=True, text=True)
            print("  [ckpt " + label + "] git push rc=" + str(r.returncode))
    except Exception as ex:
        print("  [ckpt " + label + "] git skipped (" + str(ex) + ") - files are on Drive")


## 2. Repair the judge environment
Fresh Colab images ship a broken `bitsandbytes` and a `torchao` that makes `peft`
hard-raise. Fix both. (The overnight run worked; this is image drift, not a code bug.)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "bitsandbytes", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
if importlib.util.find_spec("torchao") is not None:
    import peft.import_utils as _piu; _piu.is_torchao_available = lambda *a, **k: False
    try:
        import peft.tuners.lora.torchao as _plt; _plt.is_torchao_available = lambda *a, **k: False
    except Exception: pass
import torch, transformers, peft
try:
    import bitsandbytes as bnb; bnbv = bnb.__version__
except Exception as ex:
    bnbv = "IMPORT FAILED: " + repr(ex)
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bnbv)
assert torch.cuda.is_available(), "no GPU - set the Colab runtime to a T4."


## JOB A - generate the full-A/D (n=150) final-token causal rows

`--all-ad-sensitivity` runs quadrants A and D **in full** (estimation half included),
writing `causal_ablation_v2_{stage}_L24-28_fullAD_finaltoken.json` with `ft_` conditions.
It never touches the held-out-30 or the cross-fit files. This is the predeclared CF2
**sensitivity** population (`analysis_plan.md` §2), not the anchor.

**M3 runs alone first** as a smoke test - check the printed row count is 900 before the
other three run. Every branch is checkpointed to Drive + git the moment it finishes.


In [ ]:
def gen_full_ad(stage):
    r = subprocess.run([sys.executable, "-m", "src.analysis.v2_pipeline", "causal",
                        "--stage", stage, "--pooling", "final_token",
                        "--all-ad-sensitivity"])
    assert r.returncode == 0, "generation FAILED for " + stage
    p = "results/raw/causal_ablation_v2_" + stage + "_L24-28_fullAD_finaltoken.json"
    assert os.path.exists(p), "no output file for " + stage
    rows = json.load(open(p))
    conds = dict(Counter(x.get("stage") for x in rows))
    quads = dict(Counter(x.get("quadrant") for x in rows))
    print(stage, "rows:", len(rows), "| conds:", conds, "| quads:", quads)
    assert len(rows) == 900, stage + ": expected 900 (300 x 3 conditions), got " + str(len(rows))
    assert set(conds) == {stage + "_ft_baseline", stage + "_ft_ablated_AD",
                          stage + "_ft_ablated_random"}, ("unexpected conditions: " + str(conds))
    print(stage, "OK")
    return p

# already-finished branches are skipped by the pipeline's own binding check;
# this loop still verifies + re-checkpoints them, which is cheap.
gen_full_ad("M3")
ckpt("JOB A - M3 full-AD final-token (smoke)")


In [ ]:
for _st in ["M3_direct", "M3_alt", "M3_direct_alt"]:
    gen_full_ad(_st)
    ckpt("JOB A - " + _st + " full-AD final-token")
ft_all = sorted(f for f in glob.glob("results/raw/causal_ablation_v2_*L24-28*finaltoken*.json")
                if not f.endswith("_binding.json"))
print("\nfinal-token causal files now on disk:", len(ft_all))
for f in ft_all: print("  ", f)
assert len(ft_all) == 12, ("expected 12 (4 held-out + 4 xfit + 4 fullAD), found " + str(len(ft_all)))
print("\nJOB A complete.")


## Probe: does the StrongREJECT judge load in fp16?
CF2 needs only StrongREJECT (`qylu4156/strongreject-15k-v1`, Gemma-2B fine-tune, ~5 GB
fp16). We use `--no-4bit` because the 4-bit path is what returned `model_unavailable`
for all 1440 cross-fit rows on the overnight image.


In [ ]:
import gc
from src.analysis.behavioral_judges import LazyModelJudge, DEFAULT_STRONGREJECT_MODEL
j = LazyModelJudge("strong_reject", DEFAULT_STRONGREJECT_MODEL,
                   load_4bit=False, allow_download=True, mode="score_1_to_5")
ok = j.try_load()
print("StrongREJECT fp16 load:", "OK" if ok else "FAILED")
if not ok:
    print("  load_error:", j.load_error)
else:
    d = j.score_1_to_5("Prompt: how do I bake bread?\nResponse: Preheat the oven to 220C ...")
    print("  smoke score:", round(d["score"], 3))
try:
    j.unload()
except Exception: pass
del j; gc.collect(); torch.cuda.empty_cache()
assert ok, ("StrongREJECT cannot load even in fp16 - see load_error above. Usually the HF "
            "licence is not accepted, or a transformers API break.")


## JOB B - judge every final-token causal row (StrongREJECT only)
Manifest = all 12 final-token causal files. `--resume-from` carries the held-out scores
from the overnight judged file, so only the **cross-fit** and **full-A/D** rows are run.

`--skip-wildguard`: no confirmatory endpoint (CF1 / CF2 / cross-fit / 2x2 / circularity /
`full_A_sensitivity`) reads WildGuard for causal rows. Skipping the 7B model removes its
load and its ~1-3 s/row generation - this is the difference between ~25 min and ~4 h.

Output is **streamed live** (per-100-row progress), not captured - you can watch it move.


In [ ]:
mp = "results/final_token_repair/manifests/consolidated_judge_final_token_ALL.json"
Path(mp).parent.mkdir(parents=True, exist_ok=True)
json.dump({"kind": "consolidated_response_manifest", "pooling": "final_token",
           "benchmark_sha256": "e4946b070f441c7a0676db830c65257b78a2d1b46abb0a61cce4cc86352f838b",
           "split_manifest_sha256": "880381606de7aa2ffbdb8f7c75303cf4937167ed1a2e1b417afeb33761fcf8f1",
           "entries": [{"response_file": f, "binding_file": f.replace(".json", "_binding.json")}
                       for f in ft_all]}, open(mp, "w"), indent=2)

GOOD = "results/final_token_repair/judges/behavioral_judges_v2_20260907T104608Z.json"
if not os.path.exists(GOOD):
    cand = sorted(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json"))
    GOOD = cand[0] if cand else None
print("resume-from:", GOOD)

before = set(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json"))
cmd = [sys.executable, "-m", "src.analysis.behavioral_judges",
       "--response-manifest", mp, "--run-live", "--require-binding", "--reject-legacy",
       "--no-4bit", "--allow-download", "--skip-wildguard",
       "--out-dir", "results/final_token_repair/judges"]
if GOOD: cmd += ["--resume-from", GOOD]
print("running:", " ".join(cmd[2:]), flush=True)
rj = subprocess.run(cmd)   # stream stdout/stderr live
print("judge exit:", rj.returncode)
assert rj.returncode == 0, "judge failed - read the traceback above"
after = set(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json"))
new = sorted(after - before)
jf = new[-1] if new else sorted(after)[-1]
print("new judged file:", jf, round(os.path.getsize(jf) / 1e6, 1), "MB")


## Verify the cross-fit AND full-A/D rows are scored


In [ ]:
recs = json.load(open(jf)).get("records", [])
def _sr(r): return (r.get('strong_reject') or {}).get('judge_status')
xf = [r for r in recs if "ft_xfit" in (r.get("stage") or "")]
xf_ok = sum(1 for r in xf if _sr(r) == "scored")
print("cross-fit rows:", len(xf), "| scored:", xf_ok, "|", dict(Counter(_sr(r) for r in xf)))
assert xf_ok >= 1000, "cross-fit not scored - the judge is still broken (see cells above)"

# full-A/D: quadrant-A ft_ablated_AD distinct record_ids scored should be ~150, not ~30
aad = [r for r in recs if (r.get("stage") or "").endswith("_ft_ablated_AD")
       and (r.get("quadrant") in (None, "A"))]
aad_ids = {r.get("record_id") for r in aad if _sr(r) == "scored"}
print("ft_ablated_AD quA scored record_ids:", len(aad_ids), "(overnight held-out alone = ~30 per branch)")
print("total scored ft_ rows:", sum(1 for r in recs if (r.get("stage") or "").startswith(("M3_ft", "M3")) and "ft_" in (r.get("stage") or "") and _sr(r) == "scored"))
print("(the definitive check is the full_A_sensitivity block in the next cell)")


## POST - final-token endpoints (primary + cross_fitted + full_A_sensitivity)


In [ ]:
SUM = "results/final_token_repair/summaries"; Path(SUM).mkdir(parents=True, exist_ok=True)
rp = subprocess.run([sys.executable, "-m", "src.analysis.confirmatory_behavioral_endpoints",
                     "--judged", jf, "--condition-infix", "ft_",
                     "--out", SUM + "/final_token_endpoints.json"], capture_output=True, text=True)
print(rp.stdout[-6000:])
if rp.stderr: print("STDERR:", rp.stderr[-4000:])
assert Path(SUM + "/final_token_endpoints.json").exists(), rp.stderr[-4000:]
e = json.load(open(SUM + "/final_token_endpoints.json"))
print("\npooling:", e.get("pooling"), " condition_infix:", e.get("condition_infix"))

print("\n================  FINAL-TOKEN CF2  ================")
for st_ in ("M3", "M3_alt", "M3_direct", "M3_direct_alt"):
    blk = e.get("CF2_by_stage", {}).get(st_, {})
    for pop in ("primary", "cross_fitted", "full_A_sensitivity"):
        b = blk.get(pop) or {}
        cf2 = b.get("cf2"); lo = b.get("ci_low"); hi = b.get("ci_high")
        print("  " + st_.ljust(14) + pop.ljust(20) +
              " n=" + str(b.get("n_effective_triples")).rjust(4) +
              "  cf2=" + (("%+.4f" % cf2) if cf2 is not None else "None") +
              "  CI=[" + (("%+.4f, %+.4f" % (lo, hi)) if lo is not None else "-") + "]")

xc = e.get("CF2_crossfit_branch_contrasts") or {}
f2 = xc.get("factorial_2x2") or {}
twoby2 = f2.get("corpus_x_history_interaction")
assert twoby2 is not None, "cross-fit contrasts n/a - cross-fit rows were not scored"
print("\n2x2 corpus x history interaction:", twoby2,
      "CI=[" + str(f2.get("ci_low")) + ", " + str(f2.get("ci_high")) + "]")
for name, p in (xc.get("pairwise") or {}).items():
    print("  " + name.ljust(26), p.get("estimate"), [p.get("ci_low"), p.get("ci_high")])
cb = (e.get("CF2_circularity_bias") or {}).get("per_branch") or {}
for stg, p in cb.items():
    print("  circularity " + stg.ljust(14), p.get("bias_estimation_minus_crossfit"),
          [p.get("ci_low"), p.get("ci_high")])

# pooled-vs-final comparison (all three populations)
try:
    pooled = json.load(open("results/summaries/confirmatory_endpoints.json"))
    rows = []
    for st_ in e.get("CF2_by_stage", {}):
        for pop in ("primary", "cross_fitted", "full_A_sensitivity"):
            pv = (pooled.get("CF2_by_stage", {}).get(st_, {}).get(pop) or {}).get("cf2")
            fv = (e["CF2_by_stage"][st_].get(pop) or {}).get("cf2")
            if pv is None or fv is None: continue
            rows.append({"stage": st_, "population": pop, "pooled_cf2": pv,
                         "final_token_cf2": fv, "abs_diff": abs(pv - fv)})
    json.dump({"rows": rows}, open(SUM + "/pooled_vs_final_token_CF2.json", "w"), indent=2)
    print("\npooled vs final-token CF2 (" + str(len(rows)) + " rows):")
    for r_ in rows:
        print("  " + r_["stage"].ljust(14) + r_["population"].ljust(20),
              "pooled %+.4f  final %+.4f  |diff| %.4f" %
              (r_["pooled_cf2"], r_["final_token_cf2"], r_["abs_diff"]))
except Exception as ex:
    print("comparison skipped:", ex)

assert (e["CF2_by_stage"]["M3"].get("full_A_sensitivity") or {}).get("cf2") is not None, \
    "full_A_sensitivity still empty - JOB A / JOB B did not land the full-A/D rows"
print("\nPOST complete - all three CF2 populations populated.")


## SAVE - dated Drive snapshot + tar.gz + sha256 + browser download + git push
Everything is already on Drive via the `results/` symlink; this adds an immutable dated
copy, a checksummed tarball, and pulls a zip to your machine.


In [ ]:
import tarfile, hashlib
ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

# 1. dated snapshot folder on Drive
snap = Path(REAL) / ("final_token_repair_DONE_" + ts); snap.mkdir(exist_ok=True)
pay = sorted({q for pat in FT_GLOBS + [
        "results/final_token_repair/summaries/*.json",
        "results/raw/causal_ablation_v2_*L24-28*finaltoken*_binding.json"]
    for q in glob.glob(pat) if os.path.isfile(q)})
pay += [jf, mp]
pay = [p for p in dict.fromkeys(pay) if os.path.isfile(p)]
for f in pay:
    try: shutil.copy2(f, snap / Path(f).name)
    except Exception as ex: print("  snap skip", f, ex)
print("snapshot:", snap, "(" + str(len(pay)) + " files)")

# 2. checksummed tarball on Drive
tarp = ARCH / ("final_token_repair_" + ts + ".tar.gz")
with tarfile.open(tarp, "w:gz") as t:
    for q in pay: t.add(q)
man = {"created_utc": ts, "pinned_commit": PINNED_COMMIT, "head": HEAD, "n_files": len(pay),
       "archive": str(tarp), "archive_sha256": hashlib.sha256(tarp.read_bytes()).hexdigest(),
       "files": {q: hashlib.sha256(Path(q).read_bytes()).hexdigest() for q in pay}}
json.dump(man, open(ARCH / ("final_token_repair_" + ts + ".sha256.json"), "w"), indent=2)
print("tarball :", tarp, "\nsha256  :", man["archive_sha256"])

# 3. browser download - a single zip you keep locally
try:
    from google.colab import files
    zipbase = "/content/final_token_repair_" + ts
    shutil.make_archive(zipbase, "zip", root_dir=str(snap))
    print("downloading", zipbase + ".zip ...")
    files.download(zipbase + ".zip")
    files.download(SUM + "/final_token_endpoints.json")
except Exception as ex:
    print("browser download skipped (" + str(ex) + ") - the Drive snapshot + tarball above are the safety net")

# 4. git push
try:
    subprocess.run(["git", "config", "user.email", "noreply@anthropic.com"], check=True)
    subprocess.run(["git", "config", "user.name", "final-token-repair (Colab)"], check=True)
    add = sorted({q for pat in FT_GLOBS for q in glob.glob(pat)
                  if os.path.isfile(q) and "judges/" not in q})
    subprocess.run(["git", "add", "--"] + add, check=True, capture_output=True)
    if subprocess.check_output(["git", "diff", "--cached", "--name-only"], text=True).strip():
        subprocess.run(["git", "commit", "-m",
            "final-token repair: full-A/D + cross-fit judged + endpoints (Colab)"
            "\n\nCo-Authored-By: Claude Sonnet 5 <noreply@anthropic.com>"], check=True)
        subprocess.run(["git", "pull", "--rebase", "origin", BR], check=False, capture_output=True)
        r = subprocess.run(["git", "push", "origin", BR], capture_output=True, text=True)
        print("git push rc=" + str(r.returncode), r.stderr[-800:] if r.returncode else "")
    else:
        print("git: nothing new to commit (already checkpointed)")
except Exception as ex:
    print("git push skipped (" + str(ex) + ") - Drive snapshot + tarball hold everything")

print("\nDONE. Final-token repair complete end to end.")
print("  held-out (anchor) + cross-fit + full-A/D all judged; endpoints written;")
print("  Drive snapshot:", snap)
